In [1]:
import openai
import instructor
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient

In [2]:
from dotenv import load_dotenv

load_dotenv("../../.env")

True

In [3]:
class RAG_GenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")

In [4]:
client = instructor.from_provider("openai/gpt-5.4-nano", mode=instructor.Mode.RESPONSES_TOOLS) # can only add reasoning effort with the responses api

qdrant_client = QdrantClient( # when we are running inside of docker network 
    url="http://localhost:6333"
)

def get_embedding(text):
    response = openai.embeddings.create( # was client before
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

def retrieve_data(query, k=5): # 5 most similar items to users query
    embedding = get_embedding(query) # so we are actually creating related vector here
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=embedding,
        limit=k
    )
    
    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload.get("preprocessed_description"))
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload.get("average_rating"))
    
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }
    
def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context

def build_prompt(preprocessed_context, question): # question is the users question...
    prompt = f"""You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
"""
    return prompt

def generate_answer(prompt):
    # re write this to use instructor
    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        response_model=RAG_GenerationResponse,
        reasoning={"effort": "none"}
    )
    return response

def rag_pipeline(question, topk_k=5):
    retrievedContext = retrieve_data(question, k=topk_k)
    processed_context = process_context(retrievedContext)
    prompt = build_prompt(processed_context, question)
    answer = generate_answer(prompt) ## llm call w/ prompt
    
    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrievedContext["retrieved_context_ids"], ## surface to use for evaluation of precision and recall...
        "retrieved_context": retrievedContext["retrieved_context"],
        "similarity_scores": retrievedContext["similarity_scores"]
    }
    
    return final_answer

In [8]:
output = rag_pipeline("Do you have any earphones")

In [9]:
output["answer"]

'Yes—there are earphones available, including Jesebang Wireless Earbuds (Bluetooth 5.2, 40H playtime, Type-C charging case), Volkano Ninja Kids Wired Earphones (3.5mm, 85dB safe volume, with carry case), a 2-pack of iPhone 3.5mm wired earbuds with mic and volume control (MFi certified), Wekily Bluetooth 5.3 wireless earbuds (40H playtime, IPX7, 4-mic clear call), and a MUSICOZY Bluetooth 5.3 sports headband headphone option with built-in ENC mic.'

### RAG Pipline with grounding context (the items used to create the answer)

In [13]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="ID of item used to answer the question")
    description: str = Field(description="Description of item used to answer the question")

class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question")
    references: list[RAGUsedContext] = Field(description="List of items used to answer the the")

In [34]:
client = instructor.from_provider("openai/gpt-5.4-nano", mode=instructor.Mode.RESPONSES_TOOLS) # can only add reasoning effort with the responses api

qdrant_client = QdrantClient( # when we are running inside of docker network 
    url="http://localhost:6333"
)

def get_embedding(text):
    response = openai.embeddings.create( # was client before
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

def retrieve_data(query, k=5): # 5 most similar items to users query
    embedding = get_embedding(query) # so we are actually creating related vector here
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=embedding,
        limit=k
    )
    
    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload.get("preprocessed_description"))
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload.get("average_rating"))
    
    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }
    
def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context

def build_prompt(preprocessed_context, question): # question is the users question...
    prompt = f"""You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- If you are describing multiple products, list them out as a list

Context:
{preprocessed_context}

Question:
{question}
"""
    return prompt

def generate_answer(prompt):
    # re write this to use instructor
    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt}
        ],
        response_model=RAGGenerationResponse,
        reasoning={"effort": "none"}
    )
    return response

def rag_pipeline(question, top_k=5):
    retrievedContext = retrieve_data(question, k=top_k)
    processed_context = process_context(retrievedContext)
    prompt = build_prompt(processed_context, question)
    answer = generate_answer(prompt) ## llm call w/ prompt
    
    final_answer = {
        "data_object": answer,
        "answer": answer.answer,
        "references": answer.references,
        "question": question,
        "retrieved_context_ids": retrievedContext["retrieved_context_ids"], ## surface to use for evaluation of precision and recall...
        "retrieved_context": retrievedContext["retrieved_context"],
        "similarity_scores": retrievedContext["similarity_scores"]
    }
    
    return final_answer

In [44]:
res = rag_pipeline("Do you have any headphones", top_k = 15) # so we increaser the top k to 10 but it only used 5 of them as refernecse from the 10 retrieved context ids

In [45]:
res['references']

[RAGUsedContext(id='B09KQP2H7N', description='Kids Wireless Headphones, cat design, Bluetooth 5.0 + 3.5mm jack, adjustable/foldable, built-in microphone, up to 7 hours.'),
 RAGUsedContext(id='B0CFHWF326', description='MUSICOZY Bluetooth 5.3 Sports Headband Headphones with ENC mic, breathable fabric, 20+ hours playtime.'),
 RAGUsedContext(id='B0BM657X74', description='Bluetooth Sleep Headband / Sleep Mask Headphones with noise cancelling, thin earbuds, built-in mic and volume control, 10–12H playtime.'),
 RAGUsedContext(id='B0BRV544MV', description='Wekily Bluetooth 5.3 wireless earbuds with 40H playtime (with case), deep bass, 4-mic clear calls, IPX7 waterproof.'),
 RAGUsedContext(id='B0BBF2VC6X', description='Volkano Ninja Kids wired earbuds, 3.5mm jack, built-in mic, safe volume 85dB, includes carry case.'),
 RAGUsedContext(id='B09TFM1SFQ', description='pamu Bluetooth 5.2 ANC wireless earbuds with charging case, 4-mic noise cancellation and ENC call reduction.'),
 RAGUsedContext(id='

In [46]:
print(res['answer'])

Yes—here are the headphones available:

- Kids Wireless Headphones (cat design) — wireless/Bluetooth or 3.5mm wired, adjustable headband, foldable, built-in microphone, up to 7 hours.
- MUSICOZY Bluetooth 5.3 Headband Headphones (ENC mic) — sports headband, ENC noise reduction for calls, 20+ hours battery.
- Bluetooth Sleep Headband / Sleep Mask Headphones — thin earbuds in a stretch headband, noise canceling for sleep, 10–12H playtime.
- Wekily Bluetooth 5.3 Wireless Earbuds (LED display, 40H w/ case) — deep bass, 4-mic clear calls, IPX7 waterproof.
- Volkano Ninja Kids Wired Earbuds — 3.5mm jack, built-in mic, safe volume limit 85dB, includes carry case.
- pamu Wireless ANC Earbuds (Bluetooth 5.2) — ANC/ENC, built-in mic, charging case.
- Jesebang Bluetooth 5.2 Wireless Earbuds (40H w/ case) — touch control, built-in mic, IP7 waterproof.
- RUNAR RNR1 Wireless Neckband Running Headphones — Bluetooth 5.0 neckband, supports TF/SD memory cards.

Which type do you want (kids, sleep, wired

In [47]:
print(res['retrieved_context_ids'])

['B09KQP2H7N', 'B0CFHWF326', 'B0BM657X74', 'B0BRV544MV', 'B0BBF2VC6X', 'B09TFM1SFQ', 'B0CH8DRD6K', 'B09X9838WY', 'B0BC4PGXFK', 'B09R1S39N5', 'B0C996WY16', 'B0CC4HBS85', 'B0C4NJPN4Q', 'B0BXC72RLD', 'B0828S2KMV']


In [48]:
res

{'data_object': RAGGenerationResponse(answer='Yes—here are the headphones available:\n\n- Kids Wireless Headphones (cat design) — wireless/Bluetooth or 3.5mm wired, adjustable headband, foldable, built-in microphone, up to 7 hours.\n- MUSICOZY Bluetooth 5.3 Headband Headphones (ENC mic) — sports headband, ENC noise reduction for calls, 20+ hours battery.\n- Bluetooth Sleep Headband / Sleep Mask Headphones — thin earbuds in a stretch headband, noise canceling for sleep, 10–12H playtime.\n- Wekily Bluetooth 5.3 Wireless Earbuds (LED display, 40H w/ case) — deep bass, 4-mic clear calls, IPX7 waterproof.\n- Volkano Ninja Kids Wired Earbuds — 3.5mm jack, built-in mic, safe volume limit 85dB, includes carry case.\n- pamu Wireless ANC Earbuds (Bluetooth 5.2) — ANC/ENC, built-in mic, charging case.\n- Jesebang Bluetooth 5.2 Wireless Earbuds (40H w/ case) — touch control, built-in mic, IP7 waterproof.\n- RUNAR RNR1 Wireless Neckband Running Headphones — Bluetooth 5.0 neckband, supports TF/SD me